# Graph Laplacian Regularization in NMF-VAE

This notebook demonstrates:

1. **Basic NMF-VAE usage** without any graph regularization
2. **Graph Laplacian penalty** — what it does mathematically and intuitively
3. **STRING network integration** — building a biologically informed graph prior
4. **Effect of λ (lambda_graph)** on gene program structure across the `none → weak → moderate → strong` regimes
5. **Gene co-expression graph** as a data-driven alternative to STRING
6. **Hybrid graph** — mixing STRING structure with data-driven co-expression

All examples use synthetic count data so the notebook runs offline without a real scRNA-seq dataset.
When a real dataset and internet access are available, drop-in replacements are shown for each step.

In [ ]:
import sys, os
# Make sure the repo root is on the path when running from the notebooks/ directory
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')  # headless; use '%matplotlib inline' magic in interactive Jupyter
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)

print('Environment OK')

---
## 1  Synthetic dataset

We simulate 400 cells × 80 genes with **four ground-truth gene programs**.  
Each program drives a distinct cluster of 20 genes; cells are drawn as random convex combinations of programs.

In [ ]:
N_CELLS  = 400
N_GENES  = 80
N_PROGRAMS = 4      # ground-truth programs
GENES_PER_PROGRAM = N_GENES // N_PROGRAMS  # 20 genes per program

# Gene names (g00 … g79)
GENE_NAMES = [f'g{i:02d}' for i in range(N_GENES)]

rng = np.random.default_rng(0)

# Ground-truth W: (genes, programs) — block diagonal, genes 0-19 → prog 0, etc.
W_true = np.zeros((N_GENES, N_PROGRAMS), dtype=np.float32)
for p in range(N_PROGRAMS):
    start, end = p * GENES_PER_PROGRAM, (p + 1) * GENES_PER_PROGRAM
    W_true[start:end, p] = rng.uniform(0.5, 2.0, size=GENES_PER_PROGRAM)

# Cell loadings: each cell draws heavily from one program (+ small background)
cell_program = rng.choice(N_PROGRAMS, size=N_CELLS)
Z_true = rng.gamma(0.1, 1.0, size=(N_CELLS, N_PROGRAMS)).astype(np.float32)
for i, p in enumerate(cell_program):
    Z_true[i, p] += rng.gamma(3.0, 1.0)

# Simulate counts via Negative Binomial
mu = Z_true @ W_true.T          # (cells, genes)
mu = np.clip(mu, 0.1, None)
theta = 5.0                     # overdispersion
p_nb = theta / (theta + mu)
X = rng.negative_binomial(theta, p_nb).astype(np.float32)

print(f'Count matrix: {X.shape}  (cells × genes)')
print(f'Mean count per cell: {X.sum(1).mean():.1f}')

---
## 2  Baseline NMF-VAE (no graph prior)

Train the model with `lambda_graph=0` (the default).  
The decoder learns gene programs purely from the count data.

In [ ]:
from model.vae import NMFVAE, fit_model

LATENT_DIM = 8   # slightly over-complete on purpose

config_base = dict(
    latent_dim   = LATENT_DIM,
    hidden_dims  = [128, 64],
    epochs       = 60,
    batch_size   = 64,
    lr           = 1e-3,
    kl_weight    = 0.3,
    use_nb       = True,
    lambda_graph = 'none',   # ← string preset 'none' (λ=0.0, not Python None)
)

model_base = fit_model(X, config=config_base)
W_base = model_base.get_gene_programs()   # (genes, latent)

print(f'W_base shape: {W_base.shape}')
print(f'Final loss:   {model_base.loss_history[-1]:.2f}')

In [ ]:
def plot_W(W, title='Decoder weight matrix W', ax=None):
    """Visualise the decoder weight matrix as a heatmap."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 4))
    im = ax.imshow(W, aspect='auto', interpolation='nearest',
                   cmap='YlOrRd', vmin=0)
    ax.set_xlabel('Latent factor')
    ax.set_ylabel('Gene')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.7)
    return ax

fig, ax = plt.subplots(figsize=(7, 4))
plot_W(W_base, title='Baseline W (λ = 0)', ax=ax)
plt.tight_layout()
plt.savefig('baseline_W.png', dpi=100)
plt.show()
print('Saved baseline_W.png')

---
## 3  Graph Laplacian penalty — intuition

The penalty is:

$$
\mathcal{L}_{\text{graph}} = \lambda \cdot \mathrm{Tr}(W^\top L W)
= \lambda \sum_{i,j} A_{ij} \|\mathbf{w}_i - \mathbf{w}_j\|^2
$$

where $A_{ij}$ is the edge weight between genes $i$ and $j$ (e.g. STRING confidence).  
When two genes are **strongly connected** ($A_{ij}$ large), the penalty is large whenever their decoder rows $\mathbf{w}_i, \mathbf{w}_j$ differ.

The `NMFVAE.laplacian_penalty` static method computes exactly this:

In [ ]:
from model.vae import NMFVAE
from utils.graph_utils import build_laplacian_from_adjacency

# Two genes connected with weight 1
A_demo = np.array([[0., 1.], [1., 0.]], dtype=np.float32)
L_demo = torch.tensor(build_laplacian_from_adjacency(A_demo, normalized=False))

W_same = torch.tensor([[1., 2.], [1., 2.]])   # identical rows  → penalty should be 0
W_diff = torch.tensor([[1., 2.], [3., 5.]])   # different rows → penalty > 0

print(f'Penalty (identical rows): {NMFVAE.laplacian_penalty(W_same, L_demo).item():.4f}  ← should be 0')
print(f'Penalty (different rows): {NMFVAE.laplacian_penalty(W_diff, L_demo).item():.4f}  ← should be > 0')

---
## 4  Building a STRING-derived graph Laplacian

### 4.1  Real usage (requires internet + `requests`)

```python
from utils.graph_utils import build_string_laplacian

# gene_names must match the columns of your count matrix
L_string = build_string_laplacian(
    genes                 = gene_names,
    confidence_threshold  = 0.7,    # keep edges with STRING score ≥ 0.7
    species_id            = 9606,   # human
    normalized            = True,   # symmetric normalised Laplacian
)
```

### 4.2  Offline simulation (used here)

We construct a synthetic STRING graph that mirrors the ground-truth block structure: genes within the same program are connected with high confidence.

In [ ]:
from utils.graph_utils import build_laplacian_from_adjacency

def make_block_adjacency(n_genes, n_programs, intra_weight=0.9, inter_weight=0.0):
    """Block-diagonal adjacency mimicking STRING block structure."""
    genes_per = n_genes // n_programs
    A = np.zeros((n_genes, n_genes), dtype=np.float32)
    for p in range(n_programs):
        s, e = p * genes_per, (p + 1) * genes_per
        A[s:e, s:e] = intra_weight
        np.fill_diagonal(A[s:e, s:e], 0.0)   # no self-loops
    if inter_weight > 0:
        for i in range(n_genes):
            for j in range(n_genes):
                if A[i, j] == 0 and i != j:
                    A[i, j] = inter_weight
    return A

A_string = make_block_adjacency(N_GENES, N_PROGRAMS, intra_weight=0.9)
L_string = torch.tensor(build_laplacian_from_adjacency(A_string, normalized=True))

# Visualise the adjacency
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(A_string, cmap='Blues', interpolation='nearest', vmin=0, vmax=1)
ax.set_title('Simulated STRING adjacency A')
ax.set_xlabel('Gene')
ax.set_ylabel('Gene')
for p in range(N_PROGRAMS):
    s = p * GENES_PER_PROGRAM
    e = (p + 1) * GENES_PER_PROGRAM
    rect = plt.Rectangle((s - 0.5, s - 0.5), e - s, e - s,
                          linewidth=2, edgecolor='tomato', facecolor='none')
    ax.add_patch(rect)
plt.tight_layout()
plt.savefig('string_adjacency.png', dpi=100)
plt.show()
print('Saved string_adjacency.png')

---
## 5  Effect of λ on gene programs

We train four models with increasing regularisation strength using the named presets:

| Preset | λ value | Behaviour |
|--------|---------|----------|
| `"none"` | 0.0 | Pure data-driven |
| `"weak"` | 0.01 | Soft STRING nudge |
| `"moderate"` | 0.10 | Aligned with STRING pathways |
| `"strong"` | 1.00 | Strongly constrained by STRING |

In [ ]:
from utils.graph_utils import LAMBDA_PRESETS, resolve_lambda

print('Available presets:', LAMBDA_PRESETS)

# You can also supply a raw float to override the preset system:
print('Custom value 0.05 →', resolve_lambda(0.05))
print('Preset "moderate" →', resolve_lambda('moderate'))

In [ ]:
models   = {}
W_by_lam = {}

for preset in ['none', 'weak', 'moderate', 'strong']:
    print(f'Training  lambda_graph="{preset}" (λ={LAMBDA_PRESETS[preset]})…', end=' ', flush=True)
    config = dict(
        latent_dim   = LATENT_DIM,
        hidden_dims  = [128, 64],
        epochs       = 60,
        batch_size   = 64,
        lr           = 1e-3,
        kl_weight    = 0.3,
        use_nb       = True,
        lambda_graph = preset,
        graph_laplacian = L_string,
    )
    m = fit_model(X, config=config)
    models[preset]   = m
    W_by_lam[preset] = m.get_gene_programs()
    print(f'loss={m.loss_history[-1]:.2f}')

In [ ]:
# ── Side-by-side decoder weight matrices ──────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, preset in zip(axes.flat, ['none', 'weak', 'moderate', 'strong']):
    lam = LAMBDA_PRESETS[preset]
    plot_W(W_by_lam[preset], title=f'λ preset = "{preset}" (λ={lam})', ax=ax)

plt.suptitle('Decoder weight matrices W at different λ values', y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig('W_lambda_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved W_lambda_comparison.png')

In [ ]:
# ── Training loss curves ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
for preset, color in zip(['none', 'weak', 'moderate', 'strong'],
                          ['steelblue', 'seagreen', 'darkorange', 'crimson']):
    lam = LAMBDA_PRESETS[preset]
    ax.plot(models[preset].loss_history, label=f'λ={lam} ("{preset}")', color=color)

ax.set_xlabel('Epoch')
ax.set_ylabel('Total loss')
ax.set_title('Training loss vs epoch for different λ values')
ax.legend()
plt.tight_layout()
plt.savefig('loss_curves.png', dpi=100)
plt.show()
print('Saved loss_curves.png')

In [ ]:
# ── Quantify smoothness: within-program variance of W rows ───────────────
def within_program_variance(W, n_programs, genes_per):
    """Average variance of W rows within each ground-truth gene cluster."""
    variances = []
    for p in range(n_programs):
        s, e = p * genes_per, (p + 1) * genes_per
        block = W[s:e, :]   # (genes_per, latent)
        variances.append(block.var(axis=0).mean())
    return np.mean(variances)

print('Within-program variance of W rows (lower → more STRING-smooth):')
print(f'  {"Preset":<12} {"λ":<8} {"Variance"}')
for preset in ['none', 'weak', 'moderate', 'strong']:
    v = within_program_variance(W_by_lam[preset], N_PROGRAMS, GENES_PER_PROGRAM)
    print(f'  {preset:<12} {LAMBDA_PRESETS[preset]:<8} {v:.5f}')

### Interpretation

As **λ increases**:
- **Within-program variance decreases** — genes in the same STRING community receive more similar decoder weights.
- **Gene programs become smoother** over the STRING network.
- **Total loss increases** slightly (the penalty term adds to the ELBO loss).

The right λ depends on how strongly you trust the prior network.  
A typical recommendation is to start with `"weak"` or `"moderate"` and evaluate downstream biological coherence.

---
## 6  Supplying a custom numeric λ

The named presets are convenient starting points, but you can pass any non-negative float directly — it always overrides the preset system.

In [ ]:
config_custom = dict(
    latent_dim      = LATENT_DIM,
    hidden_dims     = [128, 64],
    epochs          = 60,
    batch_size      = 64,
    lr              = 1e-3,
    kl_weight       = 0.3,
    lambda_graph    = 0.05,       # ← custom value between "weak" and "moderate"
    graph_laplacian = L_string,
)
model_custom = fit_model(X, config=config_custom)
print(f'lambda_graph stored on model: {model_custom.lambda_graph}')
print(f'Final loss: {model_custom.loss_history[-1]:.2f}')

---
## 7  Using NMFVAE directly (low-level API)

The `NMFVAE` constructor accepts the same arguments, giving full control:

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

tensor = torch.tensor(X, dtype=torch.float32)
dataloader = DataLoader(TensorDataset(tensor), batch_size=64, shuffle=True)

model_direct = NMFVAE(
    input_dim       = N_GENES,
    latent_dim      = LATENT_DIM,
    hidden_dims     = [128, 64],
    lambda_graph    = 'moderate',
    graph_laplacian = L_string,
)

loss_history = model_direct.fit(
    dataloader,
    epochs          = 60,
    lr              = 1e-3,
    kl_weight       = 0.3,
    kl_warmup_epochs= 10,
)

print(f'lambda_graph = {model_direct.lambda_graph}')
print(f'Final loss   = {loss_history[-1]:.2f}')

### Updating the Laplacian after construction

Use `set_graph_laplacian()` to swap in a new prior without rebuilding the model:

In [ ]:
# Build a slightly noisy version of the STRING Laplacian
A_noisy = np.clip(A_string + rng.normal(0, 0.05, A_string.shape).astype(np.float32), 0, None)
A_noisy = (A_noisy + A_noisy.T) / 2   # keep symmetric
np.fill_diagonal(A_noisy, 0)
L_noisy = torch.tensor(build_laplacian_from_adjacency(A_noisy, normalized=True))

model_direct.set_graph_laplacian(L_noisy)
print('Laplacian updated — continuing training with noisy prior…')

extra_losses = model_direct.fit(dataloader, epochs=20, lr=5e-4, kl_weight=0.3)
print(f'Final loss after update: {extra_losses[-1]:.2f}')

---
## 8  Gene co-expression graph (data-driven prior)

When no protein interaction database is available, use `build_coexpression_laplacian` to construct a kNN graph from the gene co-expression structure of your count matrix.

This gives a **data-driven prior** rather than a knowledge-based one.

In [ ]:
from utils.graph_utils import build_coexpression_laplacian

# Build Laplacian from the simulated count matrix using k=5 nearest neighbours
L_coexp = build_coexpression_laplacian(X, k=5, normalized=True)

print(f'Co-expression Laplacian shape: {L_coexp.shape}')
print(f'Eigenvalue range: [{L_coexp.float().numpy().min():.4f}, '
      f'{L_coexp.float().numpy().max():.4f}]')

# Train with data-driven prior
config_coexp = dict(
    latent_dim      = LATENT_DIM,
    hidden_dims     = [128, 64],
    epochs          = 60,
    batch_size      = 64,
    lr              = 1e-3,
    kl_weight       = 0.3,
    lambda_graph    = 'moderate',
    graph_laplacian = L_coexp,
)
model_coexp = fit_model(X, config=config_coexp)
W_coexp = model_coexp.get_gene_programs()
print(f'Final loss (co-expression prior): {model_coexp.loss_history[-1]:.2f}')

---
## 9  Hybrid graph: mixing STRING and co-expression

The hybrid Laplacian blends prior knowledge with dataset-specific structure:

$$
L_{\text{hybrid}} = \alpha \, L_{\text{STRING}} + (1-\alpha) \, L_{\text{data}}
$$

- **α = 1.0** → pure STRING prior  
- **α = 0.0** → pure co-expression prior  
- **α = 0.5** → equal mix (default)

In [ ]:
from utils.graph_utils import build_hybrid_laplacian

alpha_values = [1.0, 0.75, 0.5, 0.25, 0.0]
results_hybrid = {}

for alpha in alpha_values:
    L_hybrid = build_hybrid_laplacian(L_string, L_coexp, alpha=alpha)
    config_h = dict(
        latent_dim      = LATENT_DIM,
        hidden_dims     = [128, 64],
        epochs          = 40,
        batch_size      = 64,
        lr              = 1e-3,
        kl_weight       = 0.3,
        lambda_graph    = 'moderate',
        graph_laplacian = L_hybrid,
    )
    m = fit_model(X, config=config_h)
    W_h = m.get_gene_programs()
    v = within_program_variance(W_h, N_PROGRAMS, GENES_PER_PROGRAM)
    results_hybrid[alpha] = dict(loss=m.loss_history[-1], variance=v)
    print(f'alpha={alpha:.2f}  loss={m.loss_history[-1]:.2f}  within-prog-var={v:.5f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

alphas = sorted(results_hybrid)
losses    = [results_hybrid[a]['loss']     for a in alphas]
variances = [results_hybrid[a]['variance'] for a in alphas]

ax1.plot(alphas, losses, 'o-', color='steelblue')
ax1.set_xlabel('α (STRING weight)')
ax1.set_ylabel('Final training loss')
ax1.set_title('Training loss vs α')
ax1.invert_xaxis()

ax2.plot(alphas, variances, 's-', color='darkorange')
ax2.set_xlabel('α (STRING weight)')
ax2.set_ylabel('Within-program variance of W')
ax2.set_title('Gene program smoothness vs α\n(lower = more STRING-aligned)')
ax2.invert_xaxis()

plt.suptitle('Hybrid graph: STRING vs co-expression mixing', fontsize=12)
plt.tight_layout()
plt.savefig('hybrid_alpha_sweep.png', dpi=100)
plt.show()
print('Saved hybrid_alpha_sweep.png')

---
## 10  Visualising the Laplacian penalty across the full λ sweep

We compute the penalty `Tr(WᵀLW)` for each trained model to confirm it decreases as λ increases.

In [ ]:
print(f'  {"Preset":<12} {"λ":<8} {"Tr(WᵀLW)"}')
for preset in ['none', 'weak', 'moderate', 'strong']:
    W_t = torch.tensor(W_by_lam[preset])
    pen = NMFVAE.laplacian_penalty(W_t, L_string).item()
    print(f'  {preset:<12} {LAMBDA_PRESETS[preset]:<8} {pen:.4f}')

---
## 11  Real STRING workflow (reference)

When you have real gene data and internet access, the STRING workflow is:

```python
import scanpy as sc
from model.vae import fit_model
from utils.graph_utils import build_string_laplacian, build_hybrid_laplacian, build_coexpression_laplacian

# 1. Load your data
adata = sc.read_h5ad('my_data.h5ad')
X = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
gene_names = list(adata.var_names)   # e.g. ['TP53', 'BRCA1', ...]

# 2. Build STRING Laplacian
L_string = build_string_laplacian(
    genes                = gene_names,
    confidence_threshold = 0.7,
    species_id           = 9606,   # 10090 for mouse, etc.
    normalized           = True,
)

# (optional) 3. Build co-expression Laplacian and mix
L_coexp  = build_coexpression_laplacian(X, k=15, normalized=True)
L_hybrid = build_hybrid_laplacian(L_string, L_coexp, alpha=0.7)

# 4. Train with graph prior
model = fit_model(X, config={
    'latent_dim'      : 20,
    'epochs'          : 200,
    'lambda_graph'    : 'moderate',   # or a custom float
    'graph_laplacian' : L_hybrid,
})

Z = model.transform(X)
W = model.get_gene_programs()    # (n_genes, latent_dim)
adata.obsm['X_nmfvae'] = Z
```

### CLI equivalent

```bash
python scripts/train.py \
    --input           data/processed.h5ad \
    --output          results/ \
    --latent-dim      20 \
    --epochs          200 \
    --lambda-graph    moderate \
    --use-string-graph \
    --genes-file      data/gene_names.txt \
    --confidence-threshold 0.7 \
    --use-normalized-laplacian
```

---
## Summary

| Feature | API |
|---------|-----|
| Named λ presets | `lambda_graph='none'/'weak'/'moderate'/'strong'` |
| Custom λ float  | `lambda_graph=0.05` |
| STRING Laplacian | `build_string_laplacian(genes, confidence_threshold=0.7)` |
| Co-expression Laplacian | `build_coexpression_laplacian(X, k=10)` |
| Hybrid Laplacian | `build_hybrid_laplacian(L_string, L_data, alpha=0.5)` |
| Unnormalized L | `build_laplacian_from_adjacency(A, normalized=False)` |
| Post-hoc L swap | `model.set_graph_laplacian(L)` |
| Penalty value | `NMFVAE.laplacian_penalty(W, L)` |